<a href="https://colab.research.google.com/github/NayanaKSatheesh/E-commerce_dashboard/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
df = pd.read_csv('/content/customer_shopping_behavior.csv')
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [6]:
df.describe(include = 'all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Customer ID,3900.0,NaN,NaN,NaN,1950.5,1125.977353,1.0,975.75,1950.5,2925.25,3900.0
Age,3900.0,NaN,NaN,NaN,44.068462,15.207589,18.0,31.0,44.0,57.0,70.0
Gender,3900,2,Male,2652,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Item Purchased,3900,25,Blouse,171,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,3900,4,Clothing,1737,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Purchase Amount (USD),3900.0,NaN,NaN,NaN,59.764359,23.685392,20.0,39.0,60.0,81.0,100.0
Location,3900,50,Montana,96,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Size,3900,4,M,1755,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Color,3900,25,Olive,177,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Season,3900,4,Spring,999,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


In [8]:
# filling missing values in Review Rating column with the median rating of the product category

df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [9]:
#rechecking for null values
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


In [10]:
df.columns

Index(['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category',
       'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season',
       'Review Rating', 'Subscription Status', 'Shipping Type',
       'Discount Applied', 'Promo Code Used', 'Previous Purchases',
       'Payment Method', 'Frequency of Purchases'],
      dtype='object')

In [11]:
# Renaming columns for better readability and documentation

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')
df = df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

Qcut: qcut divides the data into bins of equal size based on the distribution of values. When q=4, it creates four quartiles. This means it finds the age values that mark the 25th, 50th (median), and 75th percentiles of your age data. Each of these four bins will contain approximately 25% of your total data points (customers).

For example, the 'Young Adult' group would contain the youngest 25% of customers, 'Adult' the next 25%, and so on.

In [12]:
# create a new column age_group
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)
df[['age','age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


In [13]:
# create new column purchase_frequency_days

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [14]:
df[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [15]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [16]:
# Dropping promo code used column

df = df.drop('promo_code_used', axis=1)

In [17]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [18]:
!pip install psycopg2-binary sqlalchemy

In [22]:
from sqlalchemy import create_engine
import urllib.parse

# Connect to PostgreSQL

username = "postgres"      # default user
password = "Nayana@2004" # the password you set during installation
host = "localhost"         # if running locally
port = "5432"              # default PostgreSQL port
database = "customer_behavior"    # the database you created in pgAdmin

# URL-encode the password to handle special characters like '@'
encoded_password = urllib.parse.quote_plus(password)

engine = create_engine(f"postgresql+psycopg2://{username}:{encoded_password}@{host}:{port}/{database}")

# Step 2: Load DataFrame into PostgreSQL
table_name = "customer"   # choose any table name
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [23]:
from sqlalchemy import create_engine

# Create an in-memory SQLite database engine
sqlite_engine = create_engine('sqlite:///my_customer_data.db')

# Step 2: Load DataFrame into SQLite
sqlite_table_name = "customer_data"   # choose any table name
df.to_sql(sqlite_table_name, sqlite_engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{sqlite_table_name}' in SQLite database 'my_customer_data.db'.")

Data successfully loaded into table 'customer_data' in SQLite database 'my_customer_data.db'.


In [24]:
# Optional: Verify the data by reading it back
import pandas as pd

# Read the data back into a new DataFrame
df_from_sqlite = pd.read_sql_table(sqlite_table_name, sqlite_engine)

print("First 5 rows of data read back from SQLite:")
display(df_from_sqlite.head())

First 5 rows of data read back from SQLite:


,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365


In [28]:
#total revenue generated by male vs female customers
import pandas as pd

sql_query = """SELECT gender, SUM(purchase_amount) AS revenue FROM customer_data GROUP BY gender;"""

# Execute the query using pandas and the sqlite_engine
revenue_by_gender = pd.read_sql_query(sql_query, sqlite_engine)

print("Total Revenue by Gender:")
print(revenue_by_gender)

Total Revenue by Gender:
   gender  revenue
0  Female    75191
1    Male   157890


In [ ]:
#Which are the top 5 products with the highest average review rating?

In [29]:
sql_query_top_products = """SELECT item_purchased, ROUND(AVG(review_rating), 2) AS "Average Product Rating" FROM customer_data GROUP BY item_purchased ORDER BY AVG(review_rating) DESC LIMIT 5;"""

top_products_by_rating = pd.read_sql_query(sql_query_top_products, sqlite_engine)

print("Top 5 products with the highest average review rating:")
display(top_products_by_rating)

Top 5 products with the highest average review rating:


,item_purchased,Average Product Rating
0,Gloves,3.86
1,Sandals,3.84
2,Boots,3.82
3,Hat,3.80
4,Skirt,3.78


In [ ]:
#Do subscribed customers spend more than non-subscribed customers

In [30]:
sql_query_subscription_spend = """SELECT subscription_status, COUNT(customer_id) AS total_customers, ROUND(AVG(purchase_amount),2) AS avg_spend, ROUND(SUM(purchase_amount),2) AS total_revenue FROM customer_data GROUP BY subscription_status;"""

subscription_spend = pd.read_sql_query(sql_query_subscription_spend, sqlite_engine)

print("Customer Spending by Subscription Status:")
display(subscription_spend)

Customer Spending by Subscription Status:


,subscription_status,total_customers,avg_spend,total_revenue
0,No,2847,59.87,170436.0
1,Yes,1053,59.49,62645.0


In [ ]:
# Are customers who are repeat buyers also likely to subscribe


In [31]:
sql_query_repeat_buyers = """SELECT subscription_status, COUNT(customer_id) AS repeat_buyers FROM customer_data WHERE previous_purchases > 5 GROUP BY subscription_status;"""

repeat_buyers_subscription = pd.read_sql_query(sql_query_repeat_buyers, sqlite_engine)

print("Repeat Buyers by Subscription Status:")
display(repeat_buyers_subscription)

Repeat Buyers by Subscription Status:


,subscription_status,repeat_buyers
0,No,2518
1,Yes,958


In [ ]:
#What is the revenue contribution of each age group?


In [32]:
sql_query_revenue_by_age_group = """SELECT age_group, SUM(purchase_amount) AS total_revenue FROM customer_data GROUP BY age_group ORDER BY total_revenue DESC;"""

revenue_by_age_group = pd.read_sql_query(sql_query_revenue_by_age_group, sqlite_engine)

print("Revenue Contribution by Age Group:")
display(revenue_by_age_group)

Revenue Contribution by Age Group:


,age_group,total_revenue
0,Young Adult,62143
1,Middle-aged,59197
2,Adult,55978
3,Senior,55763


In [33]:
# Export the cleaned DataFrame to a CSV file
df.to_csv('cleaned_customer_data.csv', index=False)

print("Cleaned data exported to 'cleaned_customer_data.csv'")

Cleaned data exported to 'cleaned_customer_data.csv'
